## Тема: Сентимент анализ корпуса текстов. 
Первым делом был выбран датасет с русскими отзывами / рецензиями / комментариями, где у каждого текста отмечена эмоционлальная окраска (нейтральная, положительная или отрицательная).
Затем осуществлялась предобработка текста - удаление пунктуации, стоп-слов, лемматизация.

In [3]:
import pandas as pd

data = pd.read_csv('/home/jupyter/datasphere/project/sentiment_dataset.csv')
data.head()

import re
import nltk
from nltk.tokenize import wordpunct_tokenize

%pip install pymorphy2
from nltk.corpus import stopwords
import pymorphy2


nltk.download('stopwords')
stop_words = set(stopwords.words('russian'))
morph = pymorphy2.MorphAnalyzer()

def preprocessing(text):
    text = text.lower()                                # нижний регистр
    text = re.sub(r"http\S+|www\S+", "", text)         # ссылки
    text = re.sub(r"@\w+|#\w+", "", text)              # упоминания и хэштеги
    text = re.sub(r"[^\w\s]", " ", text)               # пунктуация
    text = re.sub(r"\d+", "", text)                    # цифры
    text = re.sub(r"\s+", " ", text).strip()           # лишние пробелы
    text = wordpunct_tokenize(text)
    text = [word for word in text if word not in stop_words]
    text = [morph.parse(word)[0].normal_form for word in text]
    return text

data['text'] = data['text'].apply(preprocessing)
data.head()

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.1
[notice] To update, run: python3 -m pip install --upgrade pip


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,label,src
0,"[пальто, красивый, прийти, дыра, молния, проси...",0,rureviews
1,"[очень, долго, идти, заказ, ждать, новый, год,...",0,rureviews
2,"[мочь, сказать, один, брюки, нормальный, порва...",0,rureviews
3,"[доставка, быстрый, маленький, месяц, заказыва...",0,rureviews
4,"[очень, понравиться, это, платье, размер, l, п...",0,rureviews


Следующий шаг - замена каждого слова его индексом в словаре и паддинг для того, чтобы тексты имели одинаковый размер. 

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print(data['label'].value_counts())

data = data[data['text'].apply(lambda x: len(x) > 0)]

print(data['label'].value_counts())

tokenizer = Tokenizer(oov_token='<UNK>')  
tokenizer.fit_on_texts(data['text'])
#tokenizer.word_index - слова с индексами
sequences = tokenizer.texts_to_sequences(data['text'])

padded_sequences = pad_sequences(sequences, maxlen=90)
padded_sequences

2025-05-01 22:03:16.022951: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 22:03:17.125413: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-01 22:03:20.942259: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


2    96992
1    96877
0    96589
Name: label, dtype: int64
2    96947
1    96788
0    96464
Name: label, dtype: int64


array([[    0,     0,     0, ...,   841,   228,  3805],
       [    0,     0,     0, ...,    27,   278,   415],
       [    0,     0,     0, ...,  3189,     3,   260],
       ...,
       [    0,     0,     0, ..., 30089, 15912,   750],
       [    0,     0,     0, ...,   244, 12785,   831],
       [    0,     0,     0, ...,   608,    47,  1185]], dtype=int32)

In [5]:
data['label'].value_counts()

2    96947
1    96788
0    96464
Name: label, dtype: int64

Видно, что классы сбалансированы, а значит, балансировка не нужна. Далее станадартным образом данные делятся на тренировочный и тестовый загрузчики. 

In [6]:
from sklearn.model_selection import train_test_split


x_train, x_test, y_train, y_test = train_test_split(padded_sequences, data['label'], test_size=0.4)
print(len(x_train), len(x_test), len(y_train), len(y_test))

import torch
from torch.utils.data import DataLoader, TensorDataset

x_train_tensor = torch.tensor(x_train, dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.long)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

train_data = TensorDataset(x_train_tensor, y_train_tensor)
test_data = TensorDataset(x_test_tensor, y_test_tensor)

batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size)
print(len(train_loader), len(test_loader))

174119 116080 174119 116080
1361 907


In [ ]:
Модель имеет слой эмбеддинга, двунаправленный слой LSTM и линейный выходной слой.

In [8]:
import torch.nn as nn

class SentimentAnalysis(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentAnalysis, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, 
                          hidden_dim, 
                          batch_first=True, 
                          bidirectional=True)
        self.dropout = nn.Dropout(0.7)
        self.fc = nn.Linear(hidden_dim*2, 3) 

    def forward(self, x):
        x = self.embedding(x)
        _, (hn, _) = self.lstm(x)
        hn_combined = torch.cat((hn[-2], hn[-1]), dim=1)
        x = self.fc(self.dropout(hn_combined))
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentimentAnalysis(len(tokenizer.word_index) + 1, 100, 32).to(device)

In [9]:
import numpy as np

def accuracy(predictions, targets):
    classes = torch.argmax(predictions, dim=1)
    return torch.mean((classes == targets).float())

from tqdm.auto import tqdm

model.train()
all_accuracy = []
all_loss = []
def train(model, loader, criterion, optimizer, num_epoch):
    for t in tqdm(range(num_epoch)):
        epoch_loss = []
        current_accuracy = 0
        current_loss = 0
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            y_pred = model(x_batch)
            y_batch = y_batch.long()
            loss = criterion(y_pred, y_batch)
            epoch_loss.append(loss.item())
            loss.backward()       
            optimizer.step()        
            optimizer.zero_grad()  

            current_accuracy += accuracy(y_pred, y_batch) 
            current_loss += loss.item()                

        epoch_accuracy = current_accuracy / len(train_loader)
        epoch_loss = current_loss / len(train_loader)

        all_accuracy.append(epoch_accuracy)
        all_loss.append(epoch_loss)
        print('Epoch {} \t'.format(t), 'Accuracy: {}\t'.format(np.round(epoch_accuracy.item(), 6)), 'Loss: {}'.format(np.round(epoch_loss, 6)))
    return model

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
train(model, train_loader, criterion, optimizer, 15)

  7%|▋         | 1/15 [00:16<03:46, 16.17s/it]

Epoch 0 	 Accuracy: 0.556169	 Loss: 0.896934


 13%|█▎        | 2/15 [00:32<03:29, 16.09s/it]

Epoch 1 	 Accuracy: 0.646752	 Loss: 0.765185


 20%|██        | 3/15 [00:47<03:08, 15.68s/it]

...

 93%|█████████▎| 14/15 [03:33<00:15, 15.31s/it]

Epoch 13 	 Accuracy: 0.822696	 Loss: 0.443186


100%|██████████| 15/15 [03:48<00:00, 15.24s/it]

Epoch 14 	 Accuracy: 0.83239	 Loss: 0.421967


SentimentAnalysis(
  (embedding): Embedding(200938, 100)
  (lstm): LSTM(100, 32, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.7, inplace=False)
  (fc): Linear(in_features=64, out_features=3, bias=True)
)

In [10]:
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        labels = labels.to(device)
        outputs = model(data) 
        _, preds = torch.max(outputs, 1) 

        all_preds.extend(preds.cpu().numpy()) 
        all_labels.extend(labels.cpu().numpy())  

from sklearn.metrics import f1_score
print(f1_score(all_labels, all_preds, average='macro'))



0.6689760751306855


Рассмотрим примеры работы сети, для этого придумаем собственные предложения. 

In [26]:
strings = [
    'Ужасное качество и плохой пошив. Возврат.',
    'Очень приятно было посетить данное место. Кормят вкусно и недорого.',
    'Неплохо, интересно, но затянуто. В целом, пойдет.',
    'Веселый, добрый и трогательный фильм, я даже заплакала.',
    'Песня растрогала, аплодисменты стоя.',
    'Сначала соседи громко шумели, затем обнаружилось отсутствие полотенец, напоследок на кухне нашли таракана. Не рекомендую.',
    'Положили лук хотя не просили, перепутали мясо. Больше не вернемся!',
    'Скучновато, зато зал украшен.',
    'После сеанса еще долго отходил и размышлял, тяжелый труд.'
]

semantics = {
    0: 'нейтральный',
    1: 'положительный',
    2: 'отрицательный'
}

j = 0
for i in strings:
    i = preprocessing(i)
    i = tokenizer.texts_to_sequences([i])
    i = pad_sequences(i, maxlen=90)
    i = torch.tensor(i)
    i = i.to(device)
    
    with torch.no_grad():
        logits = model(i)
    predicted_class = torch.argmax(logits, dim=1).item()
    print(strings[j])
    print('Предсказание: ', semantics[predicted_class])
    print()
    j += 1
    

Ужасное качество и плохой пошив. Возврат.
Предсказание:  отрицательный

Очень приятно было посетить данное место. Кормят вкусно и недорого.
Предсказание:  положительный

Неплохо, интересно, но затянуто. В целом, пойдет.
Предсказание:  нейтральный

Веселый, добрый и трогательный фильм, я даже заплакала.
Предсказание:  положительный

Песня растрогала, аплодисменты стоя.
Предсказание:  положительный

Сначала соседи громко шумели, затем обнаружилось отсутствие полотенец, напоследок на кухне нашли таракана. Не рекомендую.
Предсказание:  отрицательный

Положили лук хотя не просили, перепутали мясо. Больше не вернемся!
Предсказание:  отрицательный

Скучновато, зато зал украшен.
Предсказание:  нейтральный

После сеанса еще долго отходил и размышлял, тяжелый труд.
Предсказание:  нейтральный

